# Fine-tune a lightweight Hugging Face model and publish it
**Colab walkthrough: banking sentiment → baseline accuracy → fine-tuning → test accuracy → public model → public web app.**

We use `distilbert/distilbert-base-uncased-finetuned-sst-2-english` (about 67 million parameters). It already recognizes positive/negative sentiment, so the baseline is meaningful. This is a small classifier, not a text-generating LLM.

The goal is better banking-domain accuracy. **Improvement is not guaranteed**; this notebook measures real results and stops publication if test accuracy does not increase. No accuracy values are pre-filled. The included synthetic examples teach the workflow, not production banking quality.

**Before starting:** choose **Runtime → Change runtime type → T4 GPU**. Create a Hugging Face account and a token with permission to create/write model and Space repositories at https://huggingface.co/settings/tokens. Add it to Colab Secrets as `HF_TOKEN`, enable notebook access, or enter it securely when prompted. Never paste the token into a code cell.

Run cells from top to bottom. Training needs a GPU for a comfortable demo; CPU works more slowly. PDFs are unnecessary: sentiment classification needs labelled sentences, so we create two TXT input files.

## 1. Install packages
These versions provide the `Trainer` API used below. If Colab requests a restart, restart the session and continue from step 2. Colab supplies PyTorch.

In [ ]:
%pip -q install transformers==4.57.1 datasets==4.3.0 accelerate==1.11.0 huggingface_hub==0.36.0 scikit-learn matplotlib safetensors

## 2. Import libraries and set a reproducible seed

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed, pipeline)

set_seed(42)
BASE_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
print("Training device:", "GPU" if torch.cuda.is_available() else "CPU")

## 3. Create labelled TXT input files
One sentence per line. `positive.txt` means label **1**, `negative.txt` means label **0**.

Each underlying scenario gets several context variations. The split is by **scenario**, so variations of one sentence never appear in both training and test data. We include resolved problems as positive and unresolved problems as negative. Some sentences deliberately contain misleading words such as “excellent” in a complaint.

You can replace the seed sentences with human-labelled, non-sensitive feedback. Plain PDFs without sentiment labels cannot directly train this classifier.

In [ ]:
positive = [
    "My complaint about the failed transfer was resolved and the money is back.",
    "The fraudulent charge was reversed and my account is secure again.",
    "The bank removed the unexpected fee after reviewing my request.",
    "My blocked card was replaced quickly and now works correctly.",
    "The delayed refund arrived and the agent explained everything clearly.",
    "The incorrect loan interest was corrected to the agreed rate.",
    "My declined payment issue was fixed and the purchase went through.",
    "The missing deposit has been credited to my account.",
    "The cash withdrawal failure was resolved with a full refund.",
    "The disputed transaction was settled in my favour.",
    "The branch staff helped me complete my account opening smoothly.",
    "The mobile banking app lets me transfer money easily.",
    "The support agent answered my questions patiently and accurately.",
    "My loan application was approved within the promised timeline.",
    "The bank sent clear alerts for every account transaction.",
    "The new debit card arrived early and activation was simple.",
    "My address update was completed on the first request.",
    "The savings account charges are transparent and reasonable.",
    "The relationship manager provided useful help with my deposit.",
    "The repayment schedule was explained clearly before I signed."
]
negative = [
    "My complaint about the failed transfer is still unresolved and the money is missing.",
    "The fraudulent charge is still on my account despite my repeated complaints.",
    "The bank refused to remove the unexpected fee or explain it.",
    "My blocked card has not been replaced and I cannot access my money.",
    "The delayed refund has not arrived and nobody can explain why.",
    "The incorrect loan interest remains on my statement after three requests.",
    "My declined payment issue is still open and support stopped replying.",
    "The missing deposit has not been credited despite my receipt.",
    "The cash withdrawal failure left me without cash or a refund.",
    "The disputed transaction was closed without any investigation.",
    "The excellent rewards do not make up for unexplained charges on my account.",
    "The friendly agent promised a refund but nothing has been credited.",
    "The attractive loan offer came with hidden charges that were never disclosed.",
    "The convenient app repeatedly fails when I try to pay my bills.",
    "The helpful brochure promised instant transfers but my payment is stuck.",
    "The branch staff made me wait for hours and ignored my request.",
    "My loan application was rejected without a clear explanation.",
    "The bank sent no alert before blocking my account.",
    "My address update was ignored and the card went to the wrong address.",
    "The repayment schedule changed without notice and support will not explain."
]
contexts = [
    "Regarding my banking experience: ", "My feedback for the bank is: ",
    "Here is my customer feedback: ", "I am writing about my account. ",
    "This happened during my recent banking interaction. ", "My latest service experience: ",
    "As a banking customer, I want to report this. ", "This is my branch service feedback. ",
    "I contacted the bank recently. ", "This is the update on my banking request. ",
    "I want the bank to know the following. ", "Here is my review of the service. "
]
os.makedirs("input_files", exist_ok=True)
rows = []
for label, sentences, filename in [(1, positive, "positive.txt"), (0, negative, "negative.txt")]:
    with open("input_files/" + filename, "w", encoding="utf-8") as file:
        for scenario, sentence in enumerate(sentences):
            for context in contexts:
                text = context + sentence
                file.write(text + "\n")
                rows.append({"text": text, "label": label, "scenario": f"{label}_{scenario}"})

# Read the actual TXT files back into the data used for training.
loaded = []
for label, filename in [(1, "positive.txt"), (0, "negative.txt")]:
    with open("input_files/" + filename, encoding="utf-8") as file:
        loaded.extend({"text": line.strip(), "label": label} for line in file if line.strip())
df = pd.DataFrame(loaded)
df["scenario"] = [row["scenario"] for row in rows]
assert df["text"].is_unique
print("Examples:", len(df), "| Independent scenarios:", df["scenario"].nunique())
display(df.sample(5, random_state=42))

## 4. Split into training (60%), validation (20%), and test (20%)
Training updates weights. Validation chooses the best epoch. Test measures the final result. The scenario split prevents nearly identical variations leaking across sets. This remains a small, templated dataset: 40 independent scenarios, not 480 independent customer experiences.

In [ ]:
groups = df[["scenario", "label"]].drop_duplicates()
train_groups, remaining = train_test_split(groups, test_size=0.4, stratify=groups.label, random_state=42)
valid_groups, test_groups = train_test_split(remaining, test_size=0.5, stratify=remaining.label, random_state=42)
train_df = df[df.scenario.isin(train_groups.scenario)].reset_index(drop=True)
valid_df = df[df.scenario.isin(valid_groups.scenario)].reset_index(drop=True)
test_df = df[df.scenario.isin(test_groups.scenario)].reset_index(drop=True)
assert set(train_df.scenario).isdisjoint(test_df.scenario)
assert set(train_df.scenario).isdisjoint(valid_df.scenario)
assert set(valid_df.scenario).isdisjoint(test_df.scenario)
print("Training:", len(train_df), "Validation:", len(valid_df), "Test:", len(test_df))
for name, frame in [("train", train_df), ("validation", valid_df), ("test", test_df)]:
    frame.to_csv(f"input_files/{name}.csv", index=False)

## 5. Load the pretrained model and tokenize
Tokenization converts text into token IDs. Padding makes examples in a batch equal in length. The pretrained model already maps **0 → NEGATIVE**, **1 → POSITIVE**.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)
assert model.config.id2label[0].upper() == "NEGATIVE"
assert model.config.id2label[1].upper() == "POSITIVE"

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

def make_dataset(frame):
    data = Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    return data.map(tokenize, batched=True, remove_columns=["text"])

train_data = make_dataset(train_df)
valid_data = make_dataset(valid_df)
test_data = make_dataset(test_df)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 6. Define accuracy and F1
**Accuracy:** correct predictions ÷ all predictions. **Macro F1:** balances precision and recall for both classes. Argmax picks the class with the largest model score.

In [ ]:
def compute_metrics(result):
    logits, labels = result
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions),
            "f1": f1_score(labels, predictions, average="macro", zero_division=0)}

## 7. Check accuracy BEFORE fine-tuning
We measure the pretrained model on the fixed test set. We also record validation accuracy to monitor adaptation. Do not change the test labels or select examples to make the result look better.

In [ ]:
baseline_args = TrainingArguments(output_dir="baseline_output", per_device_eval_batch_size=16, report_to="none")
baseline_trainer = Trainer(model=model, args=baseline_args, processing_class=tokenizer,
                           data_collator=collator, compute_metrics=compute_metrics)
baseline_valid = baseline_trainer.evaluate(valid_data)
baseline_result = baseline_trainer.predict(test_data)
baseline_predictions = baseline_result.predictions.argmax(axis=-1)
baseline_accuracy = accuracy_score(test_df.label, baseline_predictions)
print(f"Before fine-tuning: test accuracy = {baseline_accuracy:.2%}")
print(f"Before fine-tuning: validation accuracy = {baseline_valid['eval_accuracy']:.2%}")

## 8. Configure fine-tuning
We update all model weights for four epochs with a small learning rate. An **epoch** is one pass over training data. The best checkpoint is selected using **validation accuracy only**. Test data is never supplied to training.

In [ ]:
training_args = TrainingArguments(
    output_dir="training_output",
    learning_rate=2e-5,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    seed=42
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_data,
                  eval_dataset=valid_data, processing_class=tokenizer,
                  data_collator=collator, compute_metrics=compute_metrics)

## 9. Fine-tune the model

In [ ]:
trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation accuracy:", trainer.state.best_metric)

## 10. Check accuracy AFTER fine-tuning
The same test sentences and labels are used before and after. The change is in **percentage points**. We also show mistakes so you can understand the result.

In [ ]:
fine_result = trainer.predict(test_data)
fine_predictions = fine_result.predictions.argmax(axis=-1)
fine_accuracy = accuracy_score(test_df.label, fine_predictions)
comparison = pd.DataFrame({
    "Model": ["Before fine-tuning", "After fine-tuning"],
    "Test accuracy": [baseline_accuracy, fine_accuracy],
    "Test macro F1": [f1_score(test_df.label, baseline_predictions, average="macro"),
                      f1_score(test_df.label, fine_predictions, average="macro")]
})
display(comparison)
print(f"Accuracy change: {(fine_accuracy - baseline_accuracy) * 100:+.2f} percentage points")
print(classification_report(test_df.label, fine_predictions, labels=[0, 1],
                            target_names=["NEGATIVE", "POSITIVE"], zero_division=0))
comparison.plot.bar(x="Model", y="Test accuracy", ylim=(0, 1), legend=False, rot=0)
plt.ylabel("Accuracy")
plt.show()
ConfusionMatrixDisplay.from_predictions(test_df.label, fine_predictions,
                                       labels=[0, 1], display_labels=["NEGATIVE", "POSITIVE"])
plt.show()
results = test_df.copy()
results["before"] = baseline_predictions
results["after"] = fine_predictions
display(results[results.label != results.after].head(10))
results.to_csv("test_predictions.csv", index=False)

### What if accuracy does not increase?
Do not invent an improvement or keep retrying against this test set. If the baseline is already 100%, improvement is impossible on this set. Use more varied, independently labelled banking feedback; investigate validation errors; choose training settings using validation data; and reserve a **fresh test set** for the next final comparison. The synthetic test set has only eight underlying scenarios, so high accuracy is not proof of real-world reliability.

The next cell requires a measured increase for this teaching exercise. If it stops, the training/evaluation still completed, but the publication cells should not run.

In [ ]:
improved = fine_accuracy > baseline_accuracy
if not improved:
    raise RuntimeError("Test accuracy did not increase. Follow the guidance above before publication.")
print("Measured improvement confirmed. Continue to save and publish.")

## 11. Save model, tokenizer, metrics, and an honest model card
The saved folder contains the weights, configuration and tokenizer needed for reuse. Only this folder is uploaded as the model; tokens and training checkpoints are excluded.

In [ ]:
if not improved:
    raise RuntimeError("Publication requires measured accuracy improvement in this exercise.")
MODEL_FOLDER = "banking_sentiment_model"
trainer.save_model(MODEL_FOLDER)
tokenizer.save_pretrained(MODEL_FOLDER)
metrics = {"baseline_test_accuracy": float(baseline_accuracy),
           "fine_tuned_test_accuracy": float(fine_accuracy),
           "best_validation_accuracy": float(trainer.state.best_metric),
           "test_rows": len(test_df), "test_scenarios": int(test_df.scenario.nunique())}
with open(MODEL_FOLDER + "/metrics.json", "w") as file:
    json.dump(metrics, file, indent=2)
card = f"""---
language: en
license: apache-2.0
base_model: {BASE_MODEL}
pipeline_tag: text-classification
tags:
- banking
- sentiment-analysis
- educational
- synthetic-data
---
# Banking sentiment — educational fine-tuning demo
DistilBERT adapted to synthetic English banking feedback. Labels: 0 NEGATIVE, 1 POSITIVE.

## Data and evaluation
480 templated rows from 40 manually written synthetic scenarios. Scenario-disjoint,
stratified 60/20/20 split; 288 training, 96 validation and 96 test rows.
Only 8 independent scenarios in the test set. No real customer data was used.
The best epoch was selected using validation accuracy.

| Metric | Value |
|---|---:|
| Baseline test accuracy | {baseline_accuracy:.4f} |
| Fine-tuned test accuracy | {fine_accuracy:.4f} |
| Best validation accuracy | {trainer.state.best_metric:.4f} |

## Limitations
Educational demonstration, not validated for production, lending, compliance or
customer decisions. Limited English examples, binary sentiment only; no neutral class.
Template repetition makes evaluation optimistic relative to real feedback.
Performance on real banking feedback and other languages is unknown.

## Training
transformers 4.57.1; learning rate 2e-5; 4 epochs; batch size 8; seed 42.
Base model: https://huggingface.co/{BASE_MODEL}
"""
with open(MODEL_FOLDER + "/README.md", "w", encoding="utf-8") as file:
    file.write(card)
print(os.listdir(MODEL_FOLDER))

## 12. Log in securely to Hugging Face
Use a write-capable token. The notebook discovers your actual username; it does not guess it. Do not share your token. Public model files can be downloaded by anyone; uploading a model alone does **not** guarantee hosted Inference Provider support.

In [ ]:
from huggingface_hub import HfApi, login
from getpass import getpass
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass("Enter your Hugging Face write token: ")
login(token=token, add_to_git_credential=False)
api = HfApi(token=token)
username = api.whoami()["name"]
MODEL_ID = f"{username}/banking-sentiment-distilbert-demo"
SPACE_ID = f"{username}/banking-sentiment-demo"
print("Public model repository:", MODEL_ID)
print("Public Space repository:", SPACE_ID)

## 13. Upload the public model
This cell creates a **new public repository**. If that name already exists, it stops to protect an existing project; change `MODEL_ID` to a new name. Run only after the measured improvement check succeeds.

In [ ]:
if not improved:
    raise RuntimeError("Accuracy improvement check failed.")
api.create_repo(repo_id=MODEL_ID, repo_type="model", private=False, exist_ok=False)
api.upload_folder(folder_path=MODEL_FOLDER, repo_id=MODEL_ID, repo_type="model",
                  commit_message="Publish banking sentiment fine-tuning demo")
print("Model published:", f"https://huggingface.co/{MODEL_ID}")

## 14. Verify that anyone can load the public model
`token=False` checks anonymous access. This downloads your uploaded model and runs a prediction. Other users can run the same pipeline with your repository ID.

In [ ]:
public_classifier = pipeline("text-classification", model=MODEL_ID, tokenizer=MODEL_ID, token=False, device=-1)
print(public_classifier([
    "My disputed payment was refunded and support explained the resolution.",
    "The agent was polite but my refund is still missing after weeks."
]))
print("Share this model ID:", MODEL_ID)

## 15. Create a public Gradio browser app
The Space loads the public model and shows sentiment and confidence. No API key is needed by visitors. Confidence is a model score, not a guarantee of correctness. Only the app files are uploaded.

In [ ]:
os.makedirs("space_app", exist_ok=True)
app_code = f'''import gradio as gr
from transformers import pipeline

classifier = pipeline("text-classification", model="{MODEL_ID}", device=-1)

def predict(text):
    if not text.strip():
        raise gr.Error("Please enter feedback.")
    result = classifier(text, truncation=True, max_length=128)[0]
    return result["label"], round(result["score"], 4)

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=4, label="Banking feedback"),
    outputs=[gr.Textbox(label="Sentiment"), gr.Number(label="Model confidence")],
    title="Banking Sentiment — Fine-tuned DistilBERT",
    description="Educational model trained on synthetic feedback. Positive/negative only. Enter non-sensitive sample text.",
    examples=[["The unexpected fee was refunded after my complaint."],
              ["The payment failed and my money is still missing."]],
    flagging_mode="never"
)
demo.launch()
'''
with open("space_app/app.py", "w") as file:
    file.write(app_code)
with open("space_app/requirements.txt", "w") as file:
    file.write("transformers==4.57.1\ntorch>=2.2,<3\nhuggingface_hub==0.36.0\nsafetensors\n")
with open("space_app/README.md", "w") as file:
    file.write("""---
title: Banking Sentiment Demo
emoji: 🏦
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 5.49.1
python_version: '3.11'
app_file: app.py
pinned: false
license: apache-2.0
---
# Banking sentiment demo
Public educational app using a fine-tuned DistilBERT classifier.
Synthetic data; not validated for production decisions.
""")
print("Space files:", os.listdir("space_app"))

Anyone can then download and use your public model:

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=MODEL_ID,
    token=False
)

print(classifier("The bank resolved my complaint quickly."))

In [ ]:
import zipfile
with zipfile.ZipFile("banking_training_inputs_and_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in os.listdir("input_files"):
        archive.write("input_files/" + filename)
    archive.write("test_predictions.csv")
    archive.write(MODEL_FOLDER + "/metrics.json", "metrics.json")
from google.colab import files
files.download("banking_training_inputs_and_results.zip")

## What changed during fine-tuning?
1. The pretrained model recognized general sentiment before training.
2. Labelled banking examples updated its existing weights.
3. Validation selected the best checkpoint without using test labels.
4. A fixed, scenario-disjoint test set measured the accuracy change.
5. The public Hub repository shares model files; the public Space provides a browser demo.

**References:**
- Base model: https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
- Text classification: https://huggingface.co/docs/transformers/tasks/sequence_classification
- Trainer: https://huggingface.co/docs/transformers/main_classes/trainer
- Uploading: https://huggingface.co/docs/huggingface_hub/guides/upload
- Gradio Spaces: https://huggingface.co/docs/hub/spaces-sdks-gradio

This notebook has been checked for notebook structure and Python syntax. Training, measured accuracy and publishing must run in your Colab session; no execution results or deployed URLs have been fabricated.